# PneumoScan IA — Entraînement YOLOv8m-seg sur RSNA
Dataset : RSNA Pneumonia Detection Challenge (Kaggle)
Split utilisé : `train` / `test`

## 1. Vérifier que le GPU est actif

In [ ]:
# Verifie que le GPU T4 est bien actif (doit afficher True)
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Pas de GPU detecte")


## 2. Uploader le fichier d'authentification Kaggle

In [ ]:
# Ouvre une fenetre pour choisir kaggle.json depuis ton PC
from google.colab import files
uploaded = files.upload()


## 3. Configurer l'accès à l'API Kaggle

In [ ]:
# Place kaggle.json a l'endroit ou l'outil kaggle s'attend a le trouver
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json


## 4. Installer les bibliothèques nécessaires

In [ ]:
# kaggle: pour telecharger le dataset
# ultralytics: pour YOLOv8m-seg
# pydicom: pour lire les images medicales au format DICOM
!pip install -q kaggle ultralytics pydicom


## 5. Télécharger et extraire le dataset RSNA

In [ ]:
# Telecharge le zip du dataset RSNA (3.66 Go) puis l'extrait dans /content/data/rsna
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/data
!cd /content/data && unzip -q rsna-pneumonia-detection-challenge.zip -d rsna


In [ ]:
# Verifie le contenu du dossier extrait
!ls /content/data/rsna


## 6. Charger le fichier des annotations

In [ ]:
# Charge le CSV qui contient, pour chaque patient, s'il a une pneumonie
# et les coordonnees de la boite si oui (Target=1) ou pas de boite (Target=0)
import pandas as pd
labels = pd.read_csv('/content/data/rsna/stage_2_train_labels.csv')
print("Nombre de lignes :", len(labels))
labels.head()


In [ ]:
# Repartition normal / pneumonie dans le dataset
labels['Target'].value_counts()


## 7. Créer les dossiers de destination (train / test)

In [ ]:
# Cree la structure de dossiers attendue par YOLO :
# dataset/images/train, dataset/images/test, dataset/labels/train, dataset/labels/test
import os
for split in ['train', 'test']:
    os.makedirs(f'/content/dataset/images/{split}', exist_ok=True)
    os.makedirs(f'/content/dataset/labels/{split}', exist_ok=True)


## 8. Préparer la répartition train / test

In [ ]:
# Separe les patients en 80% entrainement / 20% test
# (par patient, pas par ligne, pour ne pas melanger un meme patient entre les deux)
from sklearn.model_selection import train_test_split

grouped = labels.groupby('patientId')
patient_ids = labels['patientId'].unique()
train_ids, test_ids = train_test_split(patient_ids, test_size=0.2, random_state=42)

print("Patients train :", len(train_ids))
print("Patients test  :", len(test_ids))


## 9. Fonction de conversion (DICOM → PNG + annotation YOLO)

In [ ]:
# Pour un patient donne :
# 1. lit son image DICOM et la convertit en PNG
# 2. convertit ses boites (x, y, largeur, hauteur) en polygone YOLO normalise (0-1)
# 3. sauvegarde l'image et le fichier .txt d'annotation dans le bon dossier
import pydicom
import numpy as np
from PIL import Image

def convertir_patient(patient_id, split):
    dicom_path = f'/content/data/rsna/stage_2_train_images/{patient_id}.dcm'
    dicom = pydicom.dcmread(dicom_path)
    img = dicom.pixel_array

    img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
    Image.fromarray(img).save(f'/content/dataset/images/{split}/{patient_id}.png')

    h, w = img.shape
    boxes = grouped.get_group(patient_id)

    lignes_annotation = []
    for _, row in boxes.iterrows():
        if row['Target'] == 1:
            x, y, bw, bh = row['x'], row['y'], row['width'], row['height']
            x1, y1 = x / w, y / h
            x2, y2 = (x + bw) / w, y / h
            x3, y3 = (x + bw) / w, (y + bh) / h
            x4, y4 = x / w, (y + bh) / h
            lignes_annotation.append(
                f"0 {x1:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x3:.6f} {y3:.6f} {x4:.6f} {y4:.6f}"
            )

    with open(f'/content/dataset/labels/{split}/{patient_id}.txt', 'w') as f:
        f.write('\n'.join(lignes_annotation))


## 10. Lancer la conversion sur un échantillon (200 train + 50 test)

In [ ]:
# Convertit un echantillon d'abord, pour verifier que tout fonctionne
# avant de lancer la conversion complete (qui prendrait plus longtemps)
import random
random.seed(42)

echantillon_train = random.sample(list(train_ids), 200)
echantillon_test = random.sample(list(test_ids), 50)

for pid in echantillon_train:
    convertir_patient(pid, 'train')

for pid in echantillon_test:
    convertir_patient(pid, 'test')

print("Conversion de l'echantillon terminee")


## 11. Vérifier le résultat

In [ ]:
# Doit afficher 200 et 200 (une image + un fichier d'annotation par patient)
!ls /content/dataset/images/train | wc -l
!ls /content/dataset/labels/train | wc -l


In [ ]:
# Verifie aussi le cote test (doit afficher 50 et 50)
!ls /content/dataset/images/test | wc -l
!ls /content/dataset/labels/test | wc -l


In [ ]:
# Affiche le contenu d'un fichier d'annotation, pour verifier le format
import os
premier_fichier = os.listdir('/content/dataset/labels/train')[0]
print(premier_fichier)
with open(f'/content/dataset/labels/train/{premier_fichier}') as f:
    print(f.read())


---
## 12. (Etape suivante) Convertir tout le dataset
⚠️ Cette cellule traite TOUS les patients, pas juste l'echantillon — ça prend nettement plus longtemps (potentiellement 20-40 minutes selon la charge du GPU/CPU alloue par Colab). A lancer seulement une fois que l'echantillon ci-dessus fonctionne correctement.

In [ ]:
# Reinitialise les dossiers pour repartir propre (evite de melanger echantillon + tout le dataset)
import shutil
shutil.rmtree('/content/dataset', ignore_errors=True)
for split in ['train', 'test']:
    os.makedirs(f'/content/dataset/images/{split}', exist_ok=True)
    os.makedirs(f'/content/dataset/labels/{split}', exist_ok=True)

from tqdm import tqdm

for pid in tqdm(train_ids, desc="Conversion train"):
    convertir_patient(pid, 'train')

for pid in tqdm(test_ids, desc="Conversion test"):
    convertir_patient(pid, 'test')

print("Conversion complete terminee")


## 13. Créer le fichier data.yaml (configuration attendue par YOLO)

In [ ]:
# YOLO utilise toujours le mot-cle "val" en interne pour la validation,
# meme si nos dossiers a nous s'appellent "test" - on fait juste pointer
# "val" vers notre dossier test.
data_yaml = """
train: /content/dataset/images/train
val: /content/dataset/images/test

nc: 1
names: ['pneumonie']
"""

with open('/content/dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

print(data_yaml)


## 14. Lancer l'entraînement (transfert d'apprentissage sur YOLOv8m-seg)

In [ ]:
# Charge le modele pre-entraine YOLOv8m-seg (poids generiques Ultralytics)
# et l'entraine (fine-tune) sur notre dataset de radiographies
from ultralytics import YOLO

model = YOLO('yolov8m-seg.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='pneumoscan_yolov8m_seg',
    patience=10,
)


## 15. Évaluer le modèle (métriques mAP, IoU, sensibilité)

In [ ]:
# Evalue le modele entraine sur le jeu de test
metrics = model.val()
print("mAP50 :", metrics.seg.map50)
print("mAP50-95 :", metrics.seg.map)


## 16. Télécharger le modèle entraîné sur ton PC

In [ ]:
# Le meilleur modele est sauvegarde automatiquement par Ultralytics ici :
from google.colab import files
chemin_modele = 'runs/segment/pneumoscan_yolov8m_seg/weights/best.pt'
files.download(chemin_modele)
